In [ ]:
# ETAPA 2 - PASSO 1
# Carregamento dos dados preparados

import pandas as pd
from pathlib import Path

projeto = Path.cwd()

if projeto.name == "notebooks":
    projeto = projeto.parent

arquivo = projeto / "data" / "processed" / "dataset_limpo.csv"

df = pd.read_csv(arquivo)

print("=== DADOS CARREGADOS PARA TREINAMENTO ===")
print("Registros:", df.shape[0])
print("Colunas:", df.shape[1])

display(df.head())

### 📝 Observação 01 — Carregamento dos Dados para Treinamento

importar o dataset preparado na Etapa 1 para iniciar o treinamento dos modelos de Machine Learning.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 2
# Divisão dos dados em treinamento e teste

from app.data_split import dividir_dados

X = df.drop(columns=["Y"])
y = df["Y"]

X_train, X_test, y_train, y_test = dividir_dados(X, y)

print("=== DIVISÃO DOS DADOS ===")
print("Treinamento:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição das classes no treinamento:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nDistribuição das classes no teste:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

### 📝 Observação 02 — Divisão dos Dados de Treinamento e Teste

separar os registros para treinamento e avaliação dos modelos preditivos.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 3
# Preparação do pipeline de Regressão Logística

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Variáveis categóricas nominais
colunas_categoricas = ["X2", "X3", "X4"]

# Demais variáveis
colunas_numericas = [
    coluna for coluna in X_train.columns
    if coluna not in colunas_categoricas
]

# Transformações
preprocessamento = ColumnTransformer(
    transformers=[
        (
            "numericas",
            StandardScaler(),
            colunas_numericas
        ),
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            colunas_categoricas
        )
    ]
)

# Pipeline completo
modelo_logistico = Pipeline(
    steps=[
        ("preprocessamento", preprocessamento),
        (
            "classificador",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("=== PIPELINE CONFIGURADO ===")
print("Variáveis numéricas:", len(colunas_numericas))
print("Variáveis categóricas:", len(colunas_categoricas))
print("Modelo: Regressão Logística")
print("Status: Pronto para treinamento")

### 📝 Observação 03 — Configuração do Pipeline de Machine Learning

preparar o pré-processamento e configurar
o primeiro modelo de classificação de inadimplência.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 4
# Treinamento do modelo de Regressão Logística

import time

print("=== TREINAMENTO DA REGRESSÃO LOGÍSTICA ===")

# Registrar início do treinamento
inicio = time.perf_counter()

# Treinar somente com os dados de treinamento
modelo_logistico.fit(X_train, y_train)

# Calcular duração
tempo_treinamento = time.perf_counter() - inicio

print("Modelo: Regressão Logística")
print("Registros utilizados:", len(X_train))
print("Variáveis originais:", X_train.shape[1])
print("Status: Treinamento concluído")
print(f"Tempo de treinamento: {tempo_treinamento:.2f} segundos")

# Confirmar quantidade de variáveis após transformação
total_features = len(
    modelo_logistico.named_steps[
        "preprocessamento"
    ].get_feature_names_out()
)

print("Variáveis após pré-processamento:", total_features)

### 📝 Observação 04 — Treinamento da Regressão Logística

treinar o primeiro modelo supervisionado
para previsão de inadimplência.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 5
# Validação cruzada da Regressão Logística

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

validacao = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

metricas = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

resultado_cv = cross_validate(
    modelo_logistico,
    X_train,
    y_train,
    cv=validacao,
    groups=pd.util.hash_pandas_object(X_train, index=False),
    scoring=metricas,
    n_jobs=1
)

resumo_cv = pd.DataFrame({
    "Métrica": ["Precision", "Recall", "F1-score", "ROC-AUC"],
    "Média": [
        resultado_cv["test_precision"].mean(),
        resultado_cv["test_recall"].mean(),
        resultado_cv["test_f1"].mean(),
        resultado_cv["test_roc_auc"].mean()
    ],
    "Desvio padrão": [
        resultado_cv["test_precision"].std(),
        resultado_cv["test_recall"].std(),
        resultado_cv["test_f1"].std(),
        resultado_cv["test_roc_auc"].std()
    ]
})

print("=== VALIDAÇÃO CRUZADA — REGRESSÃO LOGÍSTICA ===")
display(resumo_cv.round(4))

### 📝 Observação 05 — Validação Cruzada da Regressão Logística

avaliar o desempenho inicial do modelo
utilizando validação cruzada estratificada com 5 divisões.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 6
# Treinamento do modelo Random Forest

import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone

modelo_random_forest = Pipeline(
    steps=[
        ("preprocessamento", clone(preprocessamento)),
        (
            "classificador",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=1
            )
        )
    ]
)

print("=== TREINAMENTO RANDOM FOREST ===")

inicio = time.perf_counter()

modelo_random_forest.fit(X_train, y_train)

tempo = time.perf_counter() - inicio

print("Modelo: Random Forest")
print("Registros utilizados:", len(X_train))
print("Número de árvores: 200")
print("Profundidade máxima: 12")
print("Status: Treinamento concluído")
print(f"Tempo de treinamento: {tempo:.2f} segundos")

### 📝 Observação 06 — Treinamento do Random Forest

treinar um segundo algoritmo supervisionado
para comparar seu desempenho com a Regressão Logística.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 7
# Validação cruzada do Random Forest

import pandas as pd
from sklearn.model_selection import cross_validate

print("=== VALIDAÇÃO CRUZADA — RANDOM FOREST ===")

resultado_rf_cv = cross_validate(
    modelo_random_forest,
    X_train,
    y_train,
    cv=validacao,
    groups=pd.util.hash_pandas_object(X_train, index=False),
    scoring=metricas,
    n_jobs=1
)

resumo_rf_cv = pd.DataFrame({
    "Métrica": ["Precision", "Recall", "F1-score", "ROC-AUC"],
    "Média": [
        resultado_rf_cv["test_precision"].mean(),
        resultado_rf_cv["test_recall"].mean(),
        resultado_rf_cv["test_f1"].mean(),
        resultado_rf_cv["test_roc_auc"].mean()
    ],
    "Desvio padrão": [
        resultado_rf_cv["test_precision"].std(),
        resultado_rf_cv["test_recall"].std(),
        resultado_rf_cv["test_f1"].std(),
        resultado_rf_cv["test_roc_auc"].std()
    ]
})

display(resumo_rf_cv.round(4))

### 📝 Observação 07 — Validação Cruzada do Random Forest

avaliar o desempenho do Random Forest e
compará-lo com a Regressão Logística.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 8
# Comparação dos modelos de Machine Learning

import pandas as pd

comparacao = pd.DataFrame({
    "Métrica": ["Precision", "Recall", "F1-score", "ROC-AUC"],
    "Regressão Logística": [
        resultado_cv["test_precision"].mean(),
        resultado_cv["test_recall"].mean(),
        resultado_cv["test_f1"].mean(),
        resultado_cv["test_roc_auc"].mean()
    ],
    "Random Forest": [
        resultado_rf_cv["test_precision"].mean(),
        resultado_rf_cv["test_recall"].mean(),
        resultado_rf_cv["test_f1"].mean(),
        resultado_rf_cv["test_roc_auc"].mean()
    ]
})

comparacao["Diferença (p.p.)"] = (
    comparacao["Random Forest"] -
    comparacao["Regressão Logística"]
) * 100

print("=== COMPARAÇÃO DOS MODELOS ===")

display(
    comparacao.style.format({
        "Regressão Logística": "{:.2%}",
        "Random Forest": "{:.2%}",
        "Diferença (p.p.)": "{:+.2f}"
    })
)

### 📝 Observação 08 — Comparação dos Modelos de Machine Learning

comparar o desempenho da Regressão Logística
e do Random Forest utilizando as mesmas métricas.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 9
# Análise do limiar de classificação do Random Forest

import pandas as pd

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

# Probabilidades obtidas por validação cruzada
probabilidades_cv = cross_val_predict(
    modelo_random_forest,
    X_train,
    y_train,
    cv=validacao,
    groups=pd.util.hash_pandas_object(X_train, index=False),
    method="predict_proba",
    n_jobs=1
)[:, 1]

# Avaliar diferentes limiares
limiares = [0.20, 0.30, 0.40, 0.50, 0.60]

resultados_limiares = []

for limiar in limiares:

    previsoes = (probabilidades_cv >= limiar).astype(int)

    resultados_limiares.append({
        "Limiar": limiar,
        "Precision": precision_score(
            y_train, previsoes, zero_division=0
        ),
        "Recall": recall_score(
            y_train, previsoes, zero_division=0
        ),
        "F1-score": f1_score(
            y_train, previsoes, zero_division=0
        )
    })

tabela_limiares = pd.DataFrame(resultados_limiares)

print("=== COMPARAÇÃO DOS LIMIARES — RANDOM FOREST ===")

display(tabela_limiares.round(4))

### 📝 Observação 09 — Ajuste do Limiar de Classificação

analisar como diferentes limiares de
classificação influenciam Precision, Recall e F1-score
do modelo Random Forest.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 10
# Matriz de confusão do Random Forest

import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Limiar selecionado provisoriamente
limiar = 0.30

# Previsões obtidas na validação cruzada
previsoes_cv = (probabilidades_cv >= limiar).astype(int)

# Calcular matriz de confusão
matriz = confusion_matrix(y_train, previsoes_cv)

tn, fp, fn, tp = matriz.ravel()

print("=== MATRIZ DE CONFUSÃO — RANDOM FOREST ===")

print("Verdadeiros negativos:", tn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdadeiros positivos:", tp)

# Visualização gráfica
ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Não inadimplente", "Inadimplente"]
).plot(
    cmap="Blues",
    values_format="d"
)

plt.title("Matriz de Confusão — Random Forest")
plt.show()

In [ ]:
# ETAPA 2 - PASSO 11
# Salvamento do modelo Random Forest

from pathlib import Path
import joblib

# Identificar diretório principal
projeto = Path.cwd()

if projeto.name == "notebooks":
    projeto = projeto.parent

# Criar pasta para modelos
pasta_modelos = projeto / "models"
pasta_modelos.mkdir(parents=True, exist_ok=True)

# Salvar pipeline completo
arquivo_modelo = pasta_modelos / "random_forest.joblib"

joblib.dump(modelo_random_forest, arquivo_modelo)

# Salvar limiar provisório separadamente
arquivo_limiar = pasta_modelos / "limiar.txt"
arquivo_limiar.write_text("0.30", encoding="utf-8")

print("=== MODELO SALVO ===")
print("Modelo:", arquivo_modelo)
print("Limiar:", arquivo_limiar)
print("Status: Salvamento concluído")

### 📝 Observação 11 — Salvamento do Modelo Random Forest

armazenar o modelo treinado para reutilização
na etapa de avaliação e posteriormente na API de previsão.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 2 - PASSO 12
# Salvar o conjunto de teste final

from pathlib import Path

projeto = Path.cwd()

if projeto.name == "notebooks":
    projeto = projeto.parent

pasta_saida = projeto / "data" / "processed"
pasta_saida.mkdir(parents=True, exist_ok=True)

# Reunir variáveis preditoras e alvo
df_teste = X_test.copy()
df_teste["Y"] = y_test

arquivo_teste = pasta_saida / "dataset_teste.csv"

df_teste.to_csv(arquivo_teste, index=False)

print("=== CONJUNTO DE TESTE SALVO ===")
print("Arquivo:", arquivo_teste)
print("Registros:", df_teste.shape[0])
print("Colunas:", df_teste.shape[1])
print("Distribuição da classe Y:")
print(df_teste["Y"].value_counts().sort_index())

### 📝 Observação 12 — Exportação do Conjunto de Teste

armazenar o conjunto de teste para realizar
a avaliação final do modelo de Machine Learning.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.
